In [1]:
import lhapdf
import pyhepmc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mplhep as mh
import seaborn as sns
mh.style.use("CMS")

In [2]:
# initialization of some variables
input_file = "/Users/ramos/mg5amcnlo/nuDIS_charged/Events/run_01/tag_1_pythia8_events.hepmc.gz"
n_events = None
n_mux = 9

pdf_name = "NNPDF23_lo_as_0130_qed"
pdf_set = lhapdf.getPDFSet(pdf_name)
pdf_size = pdf_set.size-1

In [3]:
pid_conversion = {
        11: r'$e^{-}$', -11: r'$e^{+}$',
        12: r'$\nu_{e}$', -12: r'$\bar{\nu}_{e}$',
        13: r'$\mu^{-}$', -13: r'$\mu^{+}$',
        14: r'$\nu_{\mu}$', -14: r'$\bar{\nu}_{\mu}$',
        15: r'$\tau^{-}$', -15: r'$\tau^{+}$',
        16: r'$\nu_{\tau}$', -16: r'$\bar{\nu}_{\tau}$',
        22: r'$\gamma$', 
        211: r'$\pi^{+}$', -211: r'$\pi^{-}$',
        2212: r'$p$', -2212: r'$\bar{p}$',
        2112: r'$n$', -2112: r'$\bar{n}$',
        321: r'$K^{+}$', -321: r'$K^{-}$',
        130: r'$K^{0}_{L}$'
    }

In [4]:
def get_events(input_file_path):
    """ returns a list of events using pyhepmc """

    with pyhepmc.open(input_file_path, format="hepmc2") as file:
        events = list(file)
    return events

In [5]:
def mod(pN):
    return(np.sqrt(pN.px**2 + pN.py**2 + pN.pz**2))

In [6]:
def dotProd(pN1, pN2):
    return (pN1.px*pN2.px + pN1.py*pN2.py + pN1.pz*pN2.pz)

In [7]:
def dphi_mpi_pi(dphi):

    while (dphi >= np.pi):
        dphi -= 2*np.pi
    while (dphi < -np.pi):
        dphi += 2*np.pi

    return dphi

In [8]:
def deltaR(v1, v2):

    dEta = v1.rap() - v2.rap()
    dPhi = dphi_mpi_pi(v1.phi() - v2.phi())

    dR = np.sqrt(dEta**2 + dPhi**2)

    return dR

In [9]:
def read_events(kevents):
    """ 
    read events and returns two dataframes:
    events_record_df: with relevant parameters (weights, pt, etc) for all particles in the final state of each event
    events_info_df: with information about the event, such as weights and 9-point scale variations
    """
    
    events_record = []
    events_info = []
    for ievt, event in enumerate(kevents):
        weight = event.weights[0]
        electrons = []
        photons = []
        for p in event.particles:
            # keep only final state particles
            if p.status != 1:
                continue
            pid = p.pid
            pN = p.momentum
            v = p.production_vertex.position

            if pid == 11:
                electrons.append(pN)
            elif pid == 22:
                photons.append(pN)

            # saves information in dictionary format
            events_record.append({
                "event": ievt,
                "ID": pid,
                "weight": weight,
                "p_{T}": pN.pt(),
                "rapidity": pN.rap(),
                "phi": pN.phi(),
                "theta": pN.theta(),
                "E": pN.e,
                "x": v.x,
                "y": v.y,
                "z": v.z
                })
            
        # record weights and 9-point scale uncertainties for each event, using dynamical scale choice 3
        # the variation on mux follows: w_mur_muf, (e.g. "w_05_10" means mur=0.5, muf=1.0)
        # the values are hardcoded, so be mindful when dealing with other hepmc files or mg5 versions
        # in this case, the weights follow: 0 = central, 1 = nominal weight, {mux variation}, {pdf variations}, resulting in 146 elements
        
        
        electrons = sorted(electrons, key = lambda ie: ie.pt(), reverse=True)
        photons = sorted(photons, key = lambda ig: ig.pt(), reverse=True)

        if len(electrons) > 0. and len(photons) > 0:
            dR = deltaR(electrons[0], photons[0])
            dPhi = dphi_mpi_pi(electrons[0].phi() - photons[0].phi())
            dRap = electrons[0].rap() - photons[0].rap()
            cosTheta = dotProd(electrons[0], photons[0]) / (mod(electrons[0])*mod(photons[0]))
            # dR = (electrons[0].phi() - photons[0].phi())**(2) + (electrons[0].rap() - photons[0].rap())**(2)
            # print(dR)
        else:
            dR = -1
            dPhi = np.nan
            dRap = np.nan

        

        events_info.append({
            "event": ievt,
            "wgt_central": weight,
            "w_05_05": event.weights[3],
            "w_05_10": event.weights[4],
            "w_05_20": event.weights[5],
            "w_10_05": event.weights[6],
            "w_10_10": event.weights[7], 
            "w_10_20": event.weights[8],
            "w_20_05": event.weights[9],
            "w_20_10": event.weights[10],
            "w_20_20": event.weights[11],
            "dR": dR,
            "dPhi": dPhi,
            "dEta": dRap,
            "cosTheta": cosTheta
            })
            
        
        
    events_record_df = pd.DataFrame(events_record)
    events_info_df = pd.DataFrame(events_info)
    
    return events_record_df, events_info_df
    

In [10]:
def is_inside_box(x, y, z, location="center"):
    """
    Function to estimate if final state particle is produced inside FASERnu (emulsion detector)
    x, y, z: production vector coordinates
    :returns: True or False
    """
    # FASERnu dimensions (in mm)
    lx = 2.5e2 # 25 cm
    ly = 3e2 # 30 cm 
    lz = 1e3 # 1 m
    if location == "center":
        return ((x >= -lx/2) & (x <= lx/2) & 
                (y >= -ly/2) & (y <= ly/2) & 
                (z >= -lz/2) & (z <= lz/2))
    elif location == "edge":
        return ((x >= -lx/2) & (x <= lx/2) & 
                (y >= -ly/2) & (y <= ly/2) & 
                (z >= 0) & (z <= lz))
    

In [11]:
def is_inside_box2(theta, location="center"):
    """
    Function to estimate if final state particle is produced inside FASERnu (emulsion detector)
    theta: angle between particle and z axis
    :returns: True or False
    """
    # FASERnu dimensions (in mm)
    ly = 3e2 # 30 cm 
    lz = 1e3 # 1 m
    if location == "center":
        return (theta < np.atan(ly/(2*lz)))
    elif location == "edge":
        return (theta < np.atan(ly/lz))
    

In [12]:
# other observables
def event_quantity(df, pid, column=None, agg_arg="size", observable="multiplicity", include_empty=True):
    """"
    General function to obtain other quantities, such as multiplicities, averages, sum of energy, etc..
    pid: PDG code of particle
    column: column from event records datafram (could be pt, energy, rapidity or momentum components)
    agg_arg: argument for aggregation funcion, if "size", it will count rows (example of usage is multiplicity), if "subleading", it will take the second highest value of the desired observable (energy, pt). Other examples can be: "sum", "mean", etc.
    
    returns events record dataframe merged with the desired quantity
    """


    if pid is not None:
        df = df[df["ID"] == pid]

    if agg_arg == "size":
        values = df.groupby("event").size()

    elif agg_arg == "subleading":
        # takes the second largest value of the observable of interest
        values = (df.groupby("event")[column].apply(lambda x: x.nlargest(2).iloc[-1] if len(x) >= 2 else np.nan))

    else:
        values = df.groupby("event")[column].agg(agg_arg)

    out = pd.DataFrame({"event": np.arange(n_evts)})

    if include_empty:
        out[observable] = out["event"].map(values).fillna(0)
    else:
        out[observable] = out["event"].map(values)
        out = out.dropna()

    return out.merge(events_info_df, on="event")

In [13]:
def weighted_mean(df, observable="multiplicity"):

    weight_columns = ["wgt_central"] + [c for c in df.columns if c.startswith("w_")]

    means = {}

    for w in weight_columns:
        norm = np.sum(df[w])
        means[w] = np.sum(df[observable] * df[w]) / norm

    variations = np.array([means[k] for k in means if k != "wgt_central"])
    means["max"] = variations.max()
    means["min"] = variations.min()

    return means

In [14]:
events = get_events(input_file)
events_record_df, events_info_df = read_events(events)
print("Total cross-section: %.3f pb." %(events_info_df["wgt_central"].sum()/len(events_info_df["wgt_central"])))

Total cross-section: 6.095 pb.


In [15]:
n_evts = events_record_df["event"].max()+1
print("Total number of generated events: %i" %(n_evts))

Total number of generated events: 9997


In [16]:
events_record_df["inside_fasernu"] = is_inside_box(events_record_df["x"], events_record_df["y"], events_record_df["z"])

In [17]:
events_record_df["inside_fasernu2"] = is_inside_box2(events_record_df["theta"])

In [18]:
# get averaged multiplicity for filtered pids
filter_pids = [22, 11, 2212, 211, -211]
rows = []

for pid in filter_pids+[2112]:

    df = event_quantity(events_record_df, pid, "multiplicity")

    unc = weighted_mean(df)

    rows.append({
        "PID": pid,
        "central": unc["wgt_central"],
        "max_var": unc["max"],
        "min_var": unc["min"]
    })

average_multi_filtered = pd.DataFrame(rows)

In [20]:
print(events_record_df["event"].nunique(), events_record_df[events_record_df["inside_fasernu"]]["event"].nunique(),
      events_record_df[events_record_df["inside_fasernu2"]]["event"].nunique(),
      events_record_df[(events_record_df["inside_fasernu2"])&(events_record_df["inside_fasernu"])]["event"].nunique())

9997 9997 9997 9997


In [21]:
print(events_record_df[(events_record_df["ID"]==11)]["event"].nunique(), 
      events_record_df[(events_record_df["ID"]==11)&(events_record_df["inside_fasernu"])]["event"].nunique(),
      events_record_df[(events_record_df["ID"]==11)&(events_record_df["inside_fasernu2"])]["event"].nunique(),
      events_record_df[(events_record_df["ID"]==11)&(events_record_df["inside_fasernu"])&(events_record_df["inside_fasernu2"])]["event"].nunique())

9997 9997 9899 9898


In [22]:
print(events_record_df[(events_record_df["ID"]==22)]["event"].nunique(), 
      events_record_df[(events_record_df["ID"]==22)&(events_record_df["inside_fasernu"])]["event"].nunique(),
      events_record_df[(events_record_df["ID"]==22)&(events_record_df["inside_fasernu2"])]["event"].nunique(),
      events_record_df[(events_record_df["ID"]==22)&(events_record_df["inside_fasernu"])&(events_record_df["inside_fasernu2"])]["event"].nunique())

9875 9868 9603 9568


In [26]:
rows = []
df2 = (events_record_df[(events_record_df["inside_fasernu"])&(events_record_df["inside_fasernu2"])].copy())
for pid in filter_pids+[2112]:

    df = event_quantity(df2, pid, "multiplicity")

    unc = weighted_mean(df)

    rows.append({
        "PID": pid,
        "central": unc["wgt_central"],
        "max_var": unc["max"],
        "min_var": unc["min"]
    })

In [28]:
average_multi_filtered

,PID,central,max_var,min_var
0,22,8.963472,8.987969,8.927323
1,11,1.058505,1.058891,1.058130
2,2212,0.868440,0.875656,0.862589
3,211,4.147516,4.147516,4.146259
4,-211,2.973507,2.976908,2.966906
5,2112,0.466031,0.468397,0.463268


In [ ]:
pd.DataFrame(rows)

,PID,central,max_var,min_var
0,22,5.944104,5.959233,5.921960
1,11,1.026894,1.027476,1.026105
2,2212,0.346212,0.346215,0.346157
3,211,2.564275,2.569490,2.556776
4,-211,2.025289,2.029117,2.021235
5,2112,0.228901,0.230940,0.226579
